# Fitting a mechanism from a specification

This is a template. The only thing you have to change is the file name in the
next cell, and then the specification it points at.

A **fit specification** is a TOML file that says which records, which mechanism
and how to search. It describes a fit and runs nothing, so the same file drives
this notebook, the `hjcfit` command and anything else that ever wants to run
the fit:

    hjcfit check my-fit.toml     # everything except the search
    hjcfit fit   my-fit.toml     # the search too

`examples/CH82.toml` fits the CH82 sample record that ships with HJCFIT, so
this notebook runs before you have changed anything. Write a new one with
`hjcfit template -o my-fit.toml`, which is the same file with every option in
it, commented.

Needs the `[fitting]` extra: `pip install 'hjcfit[fitting]'`.

In [ ]:
SPEC = "CH82.toml"          # <- your specification

import numpy as np
from matplotlib import pyplot as plt

from scipy.integrate import trapezoid

from HJCFIT.likelihood import IdealG, QMatrix
from HJCFIT.likelihood._methods import (dwell_time_histogram, ideal_pdf,
                                        ideal_pdf_scale_factor,
                                        missed_events_pdf)
from HJCFIT.likelihood.fitspec import FitSpec
from HJCFIT.likelihood.runner import (build_mechanism, load_records, run,
                                      write_result)

spec = FitSpec.from_toml(SPEC)
print(spec)

## Check it before fitting it

This is what `hjcfit check` does, and it is worth the habit: it finds the
records, builds the mechanism with every constraint applied, and evaluates the
likelihood once at the initial guess. A misspelled rate name or a critical time
below the dead time surfaces here, in a second, rather than twenty minutes into
a search.

The free-parameter list is the other reason to look. Constraints are easy to
get wrong in a direction that still runs, and a count you did not expect is the
symptom. Put the number you expect in the specification as `nfree` and it
becomes a refusal to start rather than something to notice later.

In [ ]:
records = load_records(spec)
mec = build_mechanism(spec)

for record in records:
    print(record)

free = mec.get_free_parameter_names()
print()
print(f"{mec.k} states, {mec.kA} open, {len(mec.Rates)} rate constants")
print(f"{len(free)} free: {', '.join(free)}")
held = [r.name for r in mec.Rates if not r.is_free]
if held:
    print(f"{len(held)} fixed or constrained: {', '.join(held)}")

## Fit

`run` reads the records, builds the mechanism, searches, and reports. The
default search is `simplex_hjc`, HJCFIT's own -- the search that produced every
published result -- over the logarithms of the rate constants.

Watch two things in the output. **Evaluations that could not be computed** are
reported rather than swallowed, because a mechanism the likelihood struggles
with should be visible. And **rates that ended on a limit** are printed
separately from the fitted values: a rate outside its limits is reset before
the likelihood sees it, so such a rate is not an estimate but a statement that
the likelihood wanted to go somewhere the model forbids. SCALCS gives every
rate default limits, so this can happen in a specification that sets none.

In [ ]:
outcome = run(spec, verbose=True)
print()
print(outcome)

## Does the fit describe the record?

The estimates above say what the maximum is. They do not say whether the
mechanism describes what was recorded, and that is the question an
experimenter actually faces. The display for it is the one Sigworth and Sine
introduced: dwell times binned uniformly in log time, on a square-root
ordinate, with the density the fit predicts drawn over them.

The curves come from the **fitted** mechanism, so they are what this fit
predicts and not what any truth predicts.

**The shut-time curve has to be conditioned, and the open-time curve does
not.** This record was divided into bursts at a critical shut time of 4 ms, so
the likelihood never saw a shut time longer than that: every long shut interval
lies *between* bursts and was thrown away with them. The openings were not
touched by that, so every opening in the record is in the fit. Drawing the
apparent shut-time density over a histogram of within-burst shut times
therefore means comparing a density over all shut times with a sample of the
short ones, and the curve would sit far below the histogram for no reason
having anything to do with the fit. Dividing it by its own mass in
[t_res, t_crit] makes it the density of the shut times that were actually
fitted.

How far this matters here: the fitted mechanism's slowest predicted shut time
constant is 23 800 s -- six and a half hours -- in a record about two minutes
long. A burst fit does not identify that component and was never asked to, and
only 31.6% of the predicted shut-time density lies below t_crit.

In [ ]:
record = outcome.records[0]
mec = outcome.mec
mec.set_eff('c', record.conc)         # the Q matrix at this record's concentration
qmatrix = QMatrix(mec.Q, mec.kA)
tres, tcrit = record.tres, record.tcrit

groups = [np.asarray(g, dtype=float) for g in record.groups]
opens = np.concatenate([g[0::2] for g in groups])
shuts = np.concatenate([g[1::2] for g in groups if len(g) > 1])

apparent_open = missed_events_pdf(qmatrix, tres, shut=False)
apparent_shut = missed_events_pdf(qmatrix, tres, shut=True)
ideal_open = ideal_pdf(qmatrix, shut=False)
idealG = IdealG(qmatrix)

# The mass of the apparent shut density below tcrit: the fraction of shut times
# a burst fit can see. Integrated on the same log grid the curve is drawn on,
# because the density spans eight decades.
grid = np.logspace(np.log10(tres), np.log10(tcrit), 4000)
shut_mass = float(trapezoid(apparent_shut(grid), grid))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dwell_time_histogram(
    opens, tres, ax=axes[0], pdf=apparent_open, ideal=ideal_open,
    ideal_scale=ideal_pdf_scale_factor(tres, qmatrix.aa, idealG.initial_vectors),
    xlabel='Apparent open time (s)')
axes[0].set_title(f'{len(opens)} openings')

dwell_time_histogram(
    shuts, tres, ax=axes[1], tcrit=tcrit,
    pdf=lambda t: apparent_shut(t) / shut_mass,
    xlabel='Apparent shut time (s)')
axes[1].set_title(f'{len(shuts)} shut times within bursts '
                  f'({100 * shut_mass:.1f}% of the density)')
fig.tight_layout()

## Check the curves before believing them

A predicted curve over an observed histogram is the one plot where a wrong
curve looks like a finding about the fit rather than a bug, so it is worth one
cell to check each one against a quantity it did not come from. A density must
integrate to one over the range it is a density of, and its mean must match the
mean of the sample drawn under it.

If either line below disagrees, the curve above is wrong and the fit is not
what is in question.

In [ ]:
wide = np.logspace(np.log10(tres), 8, 20000)
open_area = float(trapezoid(apparent_open(wide), wide))
open_mean = float(trapezoid(wide * apparent_open(wide), wide))
shut_mean = float(trapezoid(grid * apparent_shut(grid), grid)) / shut_mass

print(f"open: pdf integrates to {open_area:.5f}, mean {1e3 * open_mean:.4g} ms "
      f"against {1e3 * opens.mean():.4g} ms observed")
print(f"shut: {100 * shut_mass:.2f}% of the density below tcrit, conditional "
      f"mean {1e3 * shut_mean:.4g} ms against {1e3 * shuts.mean():.4g} ms observed")

## Keep the result

`write_result` writes the estimates, the specification they came from, and the
versions of everything involved, as JSON. The specification goes into the file
on purpose: a result and the description of what produced it belong in one
place, and a number in a paper that can be traced to a file is worth more than
one that has to be argued about.

The records are not in it. They are thousands of floats, and they are in the
.scn file the specification names.

In [ ]:
write_result("result.json", outcome)

import json
print(json.dumps(outcome.provenance, indent=2))

## Your own data

Change the specification, not this notebook. `hjcfit template` writes one with
every option in it; the three that matter first are:

* **`record`** -- the path to your `.scn` file, in place of a sample name.
* **`tres` and `tcrit`** -- the dead time to impose and the critical shut time
  that divides the record. `tcrit` is what decides whether an interval starts a
  new group, and getting it wrong changes the answer, not just the display.
  Leave `tcrit` out to fit the whole record as one group, which assumes one
  channel throughout; then `vectors` must be `"equilibrium"`.
* **`[mechanism]`** -- `sample` names a mechanism from `scalcs.samples.samples`;
  `mec_file` reads a DCprogs `.mec` file instead. Give the initial guess under
  `[mechanism.rates]` by rate name -- `hjcfit check` prints every name the
  mechanism has, so a typo is a list rather than a puzzle.

**Several concentrations at once** is more than one `[[data]]` section. Each
record is evaluated at its own concentration and the logarithms are added, so
the rate constants are shared and the concentrations are not. The displays
above then want one column per record.

**Which search.** `method = "simplex"` is the default and is what to quote.
Reach for `"scipy"` when the starting point is poor: a regular simplex needs
every one of its vertices to be evaluable, and on CH82's eight parameters only
6 of 200 random starting points give a finite likelihood at all.